# 🏐 ACJEUNES — Coupe de France Jeunes

Ce notebook explore et vérifie le scraping des compétitions **Coupe de France Jeunes** (entité `ACJEUNES`).

**Problème résolu :**
- Avant : **0 match** récupéré (codes de poules fictifs, short-circuit au 1er résultat, pas de support `division=`)
- Après : **5000+ matchs** via les vrais codes (`JFX`, `CFX`, `RFX`, …) et le paramètre `division=`

**Points clés :**
- ACJEUNES utilise `division=` au lieu de `poule=` dans les URLs calendrier
- La page home (`vbspo_home.php`) contient les vrais codes, pas la page ffvb.org
- La stratégie merge-all combine toutes les sources de découverte de poules

In [ ]:
# Setup
import sys
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin, urlencode

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from pyvolley.scrapers.ffvb import FFVBScraper, PouleInfo, ScrapeContext
from pyvolley.scrapers.ffvb.utils import get_current_saison, build_calendar_url
from pyvolley.scrapers.ffvb.patterns import KNOWN_POULES

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

scraper = FFVBScraper()
saison = get_current_saison()
print(f"✅ Scraper prêt — saison {saison}")

## 1. Découverte des poules ACJEUNES

Le scraper utilise 4 stratégies de découverte :
1. **Page home** (`vbspo_home.php`) — contient les liens vers les divisions actuelles
2. **Page ffvb.org** — contient seulement JMA/JFA (peu utile)
3. **Page calendrier** — ne fonctionne pas sans code de poule
4. **Patterns connus** — codes historiques et actuels connus

In [ ]:
# Découvrir toutes les poules ACJEUNES
poules = scraper.get_poules_for_entity("ACJEUNES", saison)

df_poules = pd.DataFrame([
    {
        "Code": p.code,
        "Nom": p.nom,
        "Type": "division" if p.is_division else "poule",
        "Saison": p.saison,
    }
    for p in poules
])
print(f"📋 {len(poules)} poules/divisions ACJEUNES découvertes")
print(f"   dont {sum(1 for p in poules if p.is_division)} divisions (param division=)")
print(f"   et   {sum(1 for p in poules if not p.is_division)} poules (param poule=)")
print()
df_poules

## 2. Comptage des matchs par poule

Vérifions combien de matchs on récupère par poule/division.

In [ ]:
# Compter les matchs par poule
match_counts = []
for p in poules:
    matches = list(scraper.get_matches_for_poule(
        "ACJEUNES", p.code, saison, is_division=p.is_division
    ))
    match_counts.append({
        "Code": p.code,
        "Nom": p.nom[:50],
        "Type": "div" if p.is_division else "poule",
        "Matchs": len(matches),
        "Ex. URL": (matches[0].pdf_url[:60] + "...") if matches else "-",
    })

df_counts = pd.DataFrame(match_counts)
total = df_counts["Matchs"].sum()
actifs = df_counts[df_counts["Matchs"] > 0]
print(f"🏆 Total: {total} matchs sur {len(actifs)} poules/divisions actives")
print(f"   (sur {len(poules)} poules/divisions connues)")
print()
df_counts.sort_values("Matchs", ascending=False)

## 3. Analyse des catégories

Répartition par catégorie d'âge et par genre.

In [ ]:
from pyvolley.scrapers.ffvb.utils import detect_genre, detect_categorie

analysis = []
for row in match_counts:
    nom = row["Nom"]
    analysis.append({
        "Code": row["Code"],
        "Genre": detect_genre(nom) or "?",
        "Catégorie": detect_categorie(nom) or "?",
        "Matchs": row["Matchs"],
    })

df_analysis = pd.DataFrame(analysis)

# Pivot par genre/catégorie
if not df_analysis.empty:
    pivot = df_analysis.groupby(["Genre", "Catégorie"])["Matchs"].sum().unstack(fill_value=0)
    print("📊 Répartition des matchs par genre/catégorie :")
    print()
    display(pivot)
    print(f"\nTotal : {df_analysis['Matchs'].sum()} matchs")

## 4. Comparaison multi-saisons

Vérifions que le scraper fonctionne aussi pour les saisons passées.

In [ ]:
# Tester sur les saisons passées
saisons = ["2024/2025", "2023/2024", "2022/2023"]
season_summary = []

for s in saisons:
    ps = scraper.get_poules_for_entity("ACJEUNES", s)
    n_div = sum(1 for p in ps if p.is_division)
    n_poule = sum(1 for p in ps if not p.is_division)
    season_summary.append({
        "Saison": s,
        "Divisions": n_div,
        "Poules": n_poule,
        "Total": len(ps),
        "Codes": ", ".join(p.code for p in ps[:8]) + ("..." if len(ps) > 8 else ""),
    })

df_seasons = pd.DataFrame(season_summary)
print("📅 Poules ACJEUNES par saison :")
df_seasons

## 5. Vérification de l'architecture

Test que la nouvelle architecture fonctionne correctement pour les 3 entités nationales.

In [ ]:
# Test rapide des 3 entités nationales
entities = ["ABCCS", "ACJEUNES", "AALNV"]
arch_summary = []

for ent in entities:
    ps = scraper.get_poules_for_entity(ent, saison)
    # Compter matchs d'un échantillon (2 premières poules)
    sample_matches = 0
    for p in ps[:2]:
        ms = list(scraper.get_matches_for_poule(
            ent, p.code, saison, is_division=p.is_division
        ))
        sample_matches += len(ms)
    
    arch_summary.append({
        "Entité": ent,
        "Poules": len(ps),
        "Divisions": sum(1 for p in ps if p.is_division),
        "Matchs (sample 2)": sample_matches,
        "Statut": "✅" if sample_matches > 0 else "⚠️",
    })

df_arch = pd.DataFrame(arch_summary)
print("🏗️ Vérification architecturale :")
df_arch

## 6. Patterns connus vs découverts

Comparons les codes connus (dans `patterns.py`) avec ceux découverts dynamiquement.

In [ ]:
# Patterns connus vs découverts
known_codes = set(KNOWN_POULES.get("ACJEUNES", {}).keys())
discovered_codes = {p.code for p in poules}

only_known = known_codes - discovered_codes
only_discovered = discovered_codes - known_codes
both = known_codes & discovered_codes

print(f"📊 Codes ACJEUNES :")
print(f"   Connus (patterns.py) : {len(known_codes)}")
print(f"   Découverts (dynamique) : {len(discovered_codes)}")
print(f"   En commun : {len(both)}")
print(f"   Seulement connus (historiques?) : {len(only_known)} → {sorted(only_known)}")
print(f"   Seulement découverts (nouveaux?) : {len(only_discovered)} → {sorted(only_discovered)}")